# Exercise 1: Identifying Data, Information, and Knowledge

While the lesson provided an overview of what data was, in this exercise you'll be able to explore the concept through a hands on example, understand why it's important, and how it can be used to derive information or knowledge. The following exercises will use the latest Stack Overflow Developer Survey dataset to provide hands-on experience with these concepts.

## Step 1: Gathering Data
In this step, we will extract data from the latest 2025 Stackoverflow Survey Results. These surveys often function as a benchmark for industry experts to help them understand the latest trends, patterns, and experiences around development across industries. As per Stackoverflow itself: "the survey is the definitive report on the state of software development." Note, we have provided a small sample of 10 records in CSV format, but the full table is in Parquet format for optimal data size compression. If you want to take a peak using your favorite spreadsheet software, feel free to look at the first ten records in the CSV, as the Parquet is not easily readable without a query engine. 

For more information, including raw data files, visit: https://survey.stackoverflow.co/

Historical records are also available.

We'll start by importing the necessary libraries we need to do this, including DuckDB, an embedded Online Analytical Database (OLAP) system. Note that these tools will be covered in further detail across this course. For now, the requirement of this exercise is simply to follow along and fill in the blank variables for the queries.

In [ ]:
%%capture
%config SqlMagic.autopandas = True;
%config SqlMagic.feedback = False;
%config SqlMagic.displaycon = False;
import duckdb;
import os;
%load_ext sql
conn = duckdb.connect();
%sql conn --alias duckdb
%sql duckdb:///:memory:
%sql DROP TABLE IF EXISTS raw_survey;

Note, with the above packages imported, and the correct settings configured now, we can use the Jupyter cell magic %%sql to directly execute SQL code in cells, as opposed to using Python wrappers.

In [ ]:
%%sql
CREATE TABLE raw_survey AS SELECT * FROM '../data/survey_results_public.parquet';

Let us now query an example of the data we have extracted. We'll take a peak at the first ten records to see what's in there. 

In [ ]:
%%sql
SELECT * FROM raw_survey LIMIT 10;

Notice that this table fulfils the definition of data as we discussed in the lessons. Data is a collection of discrete or continuous values about something we are studying. We have transmitted these "pieces" of digital information from the parquet files into our OLAP database, and are now processing the rows. 


In reality, the collection of data is often influenced by the perspective of those providing or gathering it. In this case, it is influenced by the developers providing answers to the survey system itself. They may answer with biases and with a specific point of view. The survey itself may have designs that orient answers a certain way as well. Data, ultimately comes from the root meaning "as given" - understood to be values about an entity as provided by whatever system or person collected them. 

## Step 2: Extracting Information

Great - but what descriptive statements can we make about the market for software development work based on these ten first data records? What is given about the state of the profession here? What about if we look at the first thousand records? What about all of the records?

The survey claims to deliver insights and teach us about the state of software development as a practice - and yet, with around ~50,000 responses from ~180 countries on over 300 technologies - isn't it difficult to make descriptive statements about the state of software development?

This is because each row or data point represents the response of a single, individual person. However, to understand the overall software industry, we need to refine this data into knowledge. We need to set a goal to understand the practical difference between raw data, summarized information, and actionable knowledge by transforming the survey results.

Consider the common question: "What is the average salary for developers who know a specific programming language?" By taking the typical steps involved in a common data pipeline (filtering, aggregation, and summarization for example), we can start to see patterns. 

#### Task 1: Fill in the blank after GROUP BY below to start. Note, in DuckDB SQL, you can GROUP BY aliases.

In [ ]:
%%sql
SELECT
    STRING_SPLIT(LanguageHaveWorkedWith, ';') AS Language,
    MEDIAN(CAST(ConvertedCompYearly AS INT)) AS MedSalary
FROM
    raw_survey
WHERE
    LanguageHaveWorkedWith IS NOT NULL
    AND ConvertedCompYearly IS NOT NULL 
    AND ConvertedCompYearly != 'NA'
    -- YOUR CODE HERE
LIMIT 20;

Recall that information is fundamentally that which allows us to interpret and understand a studied entity or its abstraction - it is data organized and summarized so it carries meaning. Note that information sent is not always necessarily the same as information received - and that the information we derive can be different based on who or what is viewing it. For example, you may be surprised to learn about the average salaries above, or you may not - depending on your experience and existing knowledge of the technology sector. 

Despite the cleaner information above (i.e. data grouped into categories), we still don't know a lot about software engineering as a practice. Ask yourself for instance, if just by looking at this chart, you could offer guidance to a junior student of software engineering - is that something you would be comfortable doing? I don't think so. Clearly, we need to refine the information further, and synthesize it into knowledge, so that we can form beliefs, as well as justifications for our beliefs, about the profession.

## Step 3: Extracting knowledge
Knowledge is a form of awareness, an epistemic truth about the reality which we are concerned with in a given context. Information becomes knowledge when we apply it to build an understanding, and use it to answer a high-value question or to describe an entity in a context. Often, the most justified knowledge is derived through experiments and scientific method - but we just have the data at hand to look at. For the time being, let's narrow our focus to digging through our dataset, so that we can find the highest-paying languages for experienced developers in the three countries with the most responses. This provides actionable insight for a specific type of job, and we will obtain knowledge about software development career opportunities as well as the market for such skilled work.

#### Task 2: Modify the query from Part B to filter for results where Country is 'United States of America' and YearsCodingExp is between 5 and 10 years.

Tip: you can use the SQL "BETWEEN X AND Y" when working with numerical values in DuckDB.

In [ ]:
%%sql
SELECT
    STRING_SPLIT(LanguageHaveWorkedWith, ';') AS Language,
    MEDIAN(CAST(ConvertedCompYearly AS INT)) AS MedSalary,
    CAST(YearsCode AS INT) AS YearsCodingExp
FROM
    raw_survey
WHERE
    LanguageHaveWorkedWith IS NOT NULL
    AND ConvertedCompYearly IS NOT NULL
    AND ConvertedCompYearly != 'NA'
    AND YearsCode != 'NA'
    -- YOUR CODE HERE
GROUP BY
    Language,
    YearsCodingExp
ORDER BY
    YearsCodingExp DESC
LIMIT 100;

Now we see more information about the individuals who replied to the survey from the United States, and who have a specific number of years in the field. Next, we'll do the same for people with different years across experience levels, to gain knowledge and understanding about the software engineering career ladder. 

#### Task 3: Replace blanks with the correct conditional statements for the CASE expression, and the WHERE clause.  

In [ ]:
%%sql
SELECT
    STRING_SPLIT(LanguageHaveWorkedWith, ';') AS Language,
    MEDIAN(CAST(ConvertedCompYearly AS INT)) AS MedSalary,
    CAST(YearsCode AS INT) AS YearsCodingExp,
    CASE WHEN YearsCodingExp >= 0 AND YearsCodingExp <= 10 THEN ' 0 - 10'
        WHEN YearsCodingExp > 10 AND YearsCodingExp <= 25 THEN '10+ - 25'
        -- YOUR CODE HERE
        WHEN YearsCodingExp > 50 AND YearsCodingExp <= 100 THEN '50+ - 100'
    END AS YearsCodingExpBucket
FROM
    raw_survey
WHERE
    LanguageHaveWorkedWith IS NOT NULL
    AND ConvertedCompYearly IS NOT NULL
    AND ConvertedCompYearly != 'NA'
    AND YearsCode != 'NA'
    -- YOUR CODE HERE
GROUP BY
    Language,
    YearsCodingExp
ORDER BY
    YearsCodingExp DESC
LIMIT 100;

Note this still doesn't help us much. Many of the developers use multiple languages, and there are so many combinations of languages one could use, it becomes confusing for us to understand how pay is impacted by years of experience. Let's focus in on one set of commonly used languages.

#### Task 4: Fill in the blanks as part of the last piece of the WHERE clause to have only Rust users appear in the filter. Do the same afterwards, but for C++ users. 

In [ ]:
%%sql
SELECT
    'Rust' AS Language,
    MEDIAN(CAST(ConvertedCompYearly AS INT)) AS MedSalary,
    CASE WHEN CAST(YearsCode AS INT)  >= 0 AND CAST(YearsCode AS INT)  <= 10 THEN ' 0 - 10'
        WHEN CAST(YearsCode AS INT)  > 10 AND CAST(YearsCode AS INT)  <= 25 THEN '10+ - 25'
        WHEN CAST(YearsCode AS INT)  > 25 AND CAST(YearsCode AS INT)  <= 50 THEN '25+ - 50'
    END AS YearsCodingExpBucket
FROM
    raw_survey
WHERE
    LanguageHaveWorkedWith IS NOT NULL
    AND ConvertedCompYearly IS NOT NULL
    AND ConvertedCompYearly != 'NA'
    AND YearsCode != 'NA'
    AND YearsCodingExpBucket != 'None'
    AND Country = 'United States of America'
    -- YOUR CODE HERE
GROUP BY
    Language,
    YearsCodingExpBucket
ORDER BY
    YearsCodingExpBucket DESC
LIMIT 100;

In [ ]:
%%sql
SELECT
    'C++' AS Language,
    MEDIAN(CAST(ConvertedCompYearly AS INT)) AS MedSalary,
    CASE WHEN CAST(YearsCode AS INT)  >= 0 AND CAST(YearsCode AS INT)  <= 10 THEN ' 0 - 10'
        WHEN CAST(YearsCode AS INT)  > 10 AND CAST(YearsCode AS INT)  <= 25 THEN '10+ - 25'
        WHEN CAST(YearsCode AS INT)  > 25 AND CAST(YearsCode AS INT)  <= 50 THEN '25+ - 50'
    END AS YearsCodingExpBucket
FROM
    raw_survey
WHERE
    LanguageHaveWorkedWith IS NOT NULL
    AND ConvertedCompYearly IS NOT NULL
    AND ConvertedCompYearly != 'NA'
    AND YearsCode != 'NA'
    AND YearsCodingExpBucket != 'None'
    AND Country = 'United States of America'
    -- YOUR CODE HERE 
GROUP BY
    Language,
    YearsCodingExpBucket
ORDER BY
    YearsCodingExpBucket DESC
LIMIT 100;

Comparing the newer language, we see that those learning Rust through their careers experience significant salary increases as their career continues. What happens if we combine those who know both C++ and Rust?

#### Task 5: Fill in the blanks as part of the last piece of the WHERE clause to have only Rust users AND C++ users appear in the filter. 

In [ ]:
%%sql
SELECT
    'C++ and Rust' AS Language,
    MEDIAN(CAST(ConvertedCompYearly AS INT)) AS MedSalary,
    CASE WHEN CAST(YearsCode AS INT)  >= 0 AND CAST(YearsCode AS INT)  <= 10 THEN ' 0 - 10'
        WHEN CAST(YearsCode AS INT)  > 10 AND CAST(YearsCode AS INT)  <= 25 THEN '10+ - 25'
        WHEN CAST(YearsCode AS INT)  > 25 AND CAST(YearsCode AS INT)  <= 50 THEN '25+ - 50'
    END AS YearsCodingExpBucket
FROM
    raw_survey
WHERE
    LanguageHaveWorkedWith IS NOT NULL
    AND ConvertedCompYearly IS NOT NULL
    AND ConvertedCompYearly != 'NA'
    AND YearsCode != 'NA'
    AND YearsCodingExpBucket != 'None'
    AND Country = 'United States of America'
    -- YOUR CODE HERE
GROUP BY
    Language,
    YearsCodingExpBucket
ORDER BY
    YearsCodingExpBucket DESC
LIMIT 100;

We now see and understand more of the career ladder. Rust is growing language and it appears there is considerable demand for it.

How does this compare with those who work with databases?!

#### Task 6: Fill in the blanks as part of the last piece of the WHERE clause to have both PostgreSQL users OR MongoDB users appear in the filter. 

In [ ]:
%%sql
SELECT
    CASE WHEN 'MongoDB' IN STRING_SPLIT(DatabaseHaveWorkedWith, ';') AND 'PostgreSQL' IN STRING_SPLIT(DatabaseHaveWorkedWith, ';') THEN 'MongoDB & PostgreSQL'
         WHEN 'MongoDB' IN STRING_SPLIT(DatabaseHaveWorkedWith, ';') THEN 'MongoDB'
         WHEN 'PostgreSQL' IN STRING_SPLIT(DatabaseHaveWorkedWith, ';') THEN 'PostgreSQL' END AS DatabaseExp,
    MEDIAN(CAST(ConvertedCompYearly AS INT)) AS MedSalary,
    CASE WHEN CAST(YearsCode AS INT)  >= 0 AND CAST(YearsCode AS INT)  <= 10 THEN ' 0 - 10'
        WHEN CAST(YearsCode AS INT)  > 10 AND CAST(YearsCode AS INT)  <= 25 THEN '10+ - 25'
        WHEN CAST(YearsCode AS INT)  > 25 AND CAST(YearsCode AS INT)  <= 50 THEN '25+ - 50'
    END AS YearsCodingExpBucket
FROM
    raw_survey
WHERE
    DatabaseExp IS NOT NULL
    AND ConvertedCompYearly IS NOT NULL
    AND ConvertedCompYearly != 'NA'
    AND YearsCode != 'NA'
    AND YearsCodingExpBucket != 'None'
    AND Country = 'United States of America'
    -- YOUR CODE HERE
GROUP BY
    DatabaseExp,
    YearsCodingExpBucket
ORDER BY
    DatabaseExp, YearsCodingExpBucket DESC
LIMIT 100;

## Standout: 
What are some potential pitfalls of this type of analysis? We believe we have found some patterns we know to be true, but how could we have erred in coming to these conclusions?

#### This is freeform answer.